# Notebook 2: Semantic Filtering (Strategy A) & CPC Validation Audit
Loads the PATSTAT baseline Parquet file and pre-computed `all-MiniLM-L6-v2` embeddings, applies right-truncation date filtering (`< 2023-01-01`), computes seed-centroid cosine similarity (Strategy A, threshold `>= 0.65`), validates domain concentration via a CPC Subclass Migration Audit, and saves the final Stage 2 dataset.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# 1. Load Parquet baseline file
parquet_path = "flying_drones_baseline_prod.parquet"
df = pd.read_parquet(parquet_path)
print(f"Loaded raw Parquet baseline: {len(df):,} records")

# 2. Apply exact text sanitization to match saved embeddings index alignment
df['appln_title'] = df['appln_title'].fillna('').str.strip()
df['appln_abstract'] = df['appln_abstract'].fillna('').str.strip()
df['full_text'] = df['appln_title'] + ". " + df['appln_abstract']

short_mask = df['full_text'].str.strip().str.len() <= 10
df = df[~short_mask].copy().reset_index(drop=True)
print(f"Sanitized corpus ready: {len(df):,} records")

Loaded raw Parquet baseline: 219,997 records
Sanitized corpus ready: 219,996 records


In [3]:
# 3. Load pre-computed vector embeddings (384-d) & verify 1:1 alignment
corpus_embeddings = np.load("flying_drones_embeddings.npy")
assert len(df) == corpus_embeddings.shape[0], (
    f"Mismatch: {len(df)} DF rows vs {corpus_embeddings.shape[0]} embeddings"
)
print(f"Embeddings verified: {corpus_embeddings.shape}")

Embeddings verified: (219996, 384)


In [4]:
# 4. Apply date window truncation (< 2023-01-01) to BOTH df and corpus_embeddings
# Removes recent years affected by the 18-month publication & PATSTAT ingestion lag
MIN_DATE = pd.Timestamp("2000-01-01")
MAX_DATE = pd.Timestamp("2023-01-01")

filing_dates = pd.to_datetime(df['appln_filing_date'], errors='coerce')
date_mask = (filing_dates >= MIN_DATE) & (filing_dates < MAX_DATE)

df = df[date_mask].copy().reset_index(drop=True)
corpus_embeddings = corpus_embeddings[date_mask.values]

print(f"Corpus after date truncation (< {MAX_DATE.date()}): {len(df):,} records | Embeddings: {corpus_embeddings.shape}")

Corpus after date truncation (< 2023-01-01): 156,129 records | Embeddings: (156129, 384)


## Strategy A: Seed-Centroid Cosine Similarity Filtering

In [5]:
# 5. Compute Centroid Similarity (Strategy A)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

seed_candidates = df[
    df['cpc_codes'].str.contains('B64U', na=False) &
    df['appln_title'].str.contains('unmanned aerial vehicle|quadcopter|multirotor', case=False, na=False)
].head(15)

seed_embeddings = model.encode(
    seed_candidates['full_text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)
centroid_vector = np.mean(seed_embeddings, axis=0, keepdims=True)
centroid_vector = centroid_vector / np.linalg.norm(centroid_vector)

df['sim_score_centroid'] = cosine_similarity(corpus_embeddings, centroid_vector).flatten()

print("Strategy A Similarity Score Quantiles:")
print(df['sim_score_centroid'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Strategy A Similarity Score Quantiles:
0.10    0.350266
0.25    0.451731
0.50    0.537630
0.75    0.605698
0.90    0.655503
0.95    0.682926
0.99    0.727820
Name: sim_score_centroid, dtype: float64


In [6]:
# 6. Filter dataset using Strategy A high-precision threshold (>= 0.65)
SIM_THRESHOLD = 0.65
df_filtered = df[df['sim_score_centroid'] >= SIM_THRESHOLD].copy().reset_index(drop=True)

print(f"Retained {len(df_filtered):,} patents out of {len(df):,} baseline records (sim >= {SIM_THRESHOLD})")

Retained 17,689 patents out of 156,129 baseline records (sim >= 0.65)


## Validation: CPC Subclass Migration Audit

In [7]:
# 7. Extract 4-character CPC subclasses and compare pre- vs. post-filtering distributions
def extract_cpc_subclasses(cpc_series):
    all_codes = cpc_series.dropna().astype(str).str.findall(r'\b[A-H]\d{2}[A-Z]\b')
    flattened = [code for sublist in all_codes for code in set(sublist)]
    return pd.Series(flattened)

baseline_counts = extract_cpc_subclasses(df['cpc_codes']).value_counts(normalize=True) * 100
filtered_counts = extract_cpc_subclasses(df_filtered['cpc_codes']).value_counts(normalize=True) * 100

migration_df = pd.DataFrame({
    'Baseline Share (%)': baseline_counts,
    'Filtered Share (%)': filtered_counts
}).fillna(0)

migration_df['Share Change (pp)'] = migration_df['Filtered Share (%)'] - migration_df['Baseline Share (%)']
migration_df = migration_df.sort_values(by='Filtered Share (%)', ascending=False)

print("=== CPC Subclass Migration Audit (Top 15 Filtered Subclasses) ===")
print(migration_df.head(15).to_string())

=== CPC Subclass Migration Audit (Top 15 Filtered Subclasses) ===
      Baseline Share (%)  Filtered Share (%)  Share Change (pp)
B64U           22.226452           23.363491           1.137040
B64C            9.904533           18.773284           8.868751
B64D            8.618948           13.783928           5.164979
G05D            8.976252           11.291467           2.315215
B64F            2.581320            4.896221           2.314902
G08G            2.671661            2.971439           0.299778
G06Q            1.922033            1.339365          -0.582668
G01S            2.433120            1.193010          -1.240109
H04N            2.133674            1.148661          -0.985014
G01C            2.476260            1.028916          -1.447344
B60L            1.109470            0.971261          -0.138209
H04W            2.623445            0.873692          -1.749753
B60P            0.320762            0.851517           0.530755
G06V            2.244824            0.

### Audit Interpretation
Semantic filtering consolidates core aeronautical and flight-control classifications while pruning generic noise:
- **B64U (UAV):** Increased from 22.22% to 23.36% (+1.14 pp)
- **B64C (Aircraft Structures, Rotors, Aerodynamics):** Increased from 9.90% to 18.77% (+8.86 pp)
- **B64D (Aircraft Equipment, Payloads, Powerplants):**  Increased from 8.62% to 13.78% (+5.16 pp)
- **G05D (Autopilot, Flight Path & Attitude Control):** Rose from 8.97% to 11.29% (+2.31 pp)
- **B64F (Ground Stations, Launching, Recovery Docks):** More than doubled from 2.58% to 4.89% (+2.31 pp)

Together, these 5 core subclasses expand from **52%** of the baseline to **73%** of the filtered corpus, while generic computer vision (`G06V`), software (`G06F`), and wireless infrastructure (`H04W`) drop substantially.

In [8]:
# 8. Save final cleaned Stage 2 dataset to Parquet for team sharing & Stage 3 analysis
df_final = df_filtered.reset_index(drop=True)
output_parquet = "flying_drones_stage2_clean.parquet"
df_final.to_parquet(output_parquet, index=False)

size_mb = os.path.getsize(output_parquet) / (1024 * 1024)
print(f"Successfully saved '{output_parquet}': {len(df_final):,} patents ({size_mb:.2f} MB)")

Successfully saved 'flying_drones_stage2_clean.parquet': 17,689 patents (16.43 MB)
